# Module 302 — Clean Label Attack Study

HTB Academy, AI Data Attacks, Sections 9–10: **Clean Label Attacks** and its clean one-vs-rest (OvR) baseline. This notebook introduces the three-class manufacturing example, creates a reproducible synthetic dataset, makes a stratified train/test split, standardizes the features without test-set leakage, and plots the clean training data, and fits a clean three-class reference model before any poisoning.

Sections 9–10 introduce the attack idea, prepare its dataset, and establish the clean reference model. They do **not** implement feature poisoning yet; that belongs to later course material. The labels in this notebook remain clean. The two synthetic measurements stand in for component length and weight; they are not real factory data.

## Read the notation aloud

- `x_i = (x_i1, x_i2)`: **“ex sub i equals the pair ex sub i one, ex sub i two.”** This is row `i` with its two feature measurements.
- `y_i`: **“why sub i.”** This is the ground-truth class label for row `i`.
- Conceptual feature edit: `x_i′ = x_i + δ_i`, **“ex sub i prime equals ex sub i plus delta sub i.”** `x_i′` is the altered feature row and `δ_i` is the vector of small changes. Then `δ_i = x_i′ − x_i`, **“delta sub i equals ex sub i prime minus ex sub i.”**
- Clean-label condition: `y_i′ = y_i`, **“why sub i prime equals why sub i.”** The feature row changes while its assigned label stays the same. This notation is a teaching shorthand for the attack idea; no feature edits are applied in this section.


## 1. Imports and repeatable settings

NumPy and scikit-learn create and split the toy dataset. `StandardScaler` learns feature means and standard deviations from training data. `LogisticRegression` supplies each binary model, and `OneVsRestClassifier` combines them for three classes. `accuracy_score` measures the clean baseline. Matplotlib draws the classes and learned regions. The class-to-name mapping follows HTB's quality-control story: 0 is Major Defect, 1 is Acceptable, and 2 is Minor Defect.

This subsection only needs data setup and visualization. Attack-specific helpers are left for the sections that introduce them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score

SEED = 1337
CLASS_NAMES = {
    0: "Major Defect",
    1: "Acceptable",
    2: "Minor Defect",
}
CLASS_COLORS = {
    0: "#0086ff",  # blue
    1: "#ffaf00",  # orange
    2: "#ff3e3e",  # red
}


## 2. Generate the three-class feature matrix

`make_blobs` creates 1,500 points around three centers. Each row of `X_raw` is one two-measurement example, and the matching entry in `y` is its class number. The centers are separated enough that a later attack's effect should be visible in a plot.

For instance, the first feature might represent length and the second weight. Those names are only an analogy: the generated values are artificial.


In [ ]:
N_SAMPLES = 1_500
CENTERS = [(0, 6), (4, 3), (8, 6)]
CLUSTER_STD = 1.15

X_raw, y = make_blobs(
    n_samples=N_SAMPLES,
    centers=CENTERS,
    n_features=2,
    cluster_std=CLUSTER_STD,
    random_state=SEED,
)

print(f"Generated features: {X_raw.shape}  (rows, two measurements)")
print(f"Generated labels:   {y.shape}")
print(f"Class counts:       {np.bincount(y)}")
print(f"Class names:        {CLASS_NAMES}")


## 3. Split first and preserve class proportions

A 30% test fraction reserves 450 of the 1,500 examples for evaluation. `stratify=y` asks the split to keep each class's proportion similar in both partitions. With this balanced toy dataset, the expected split has 350 training and 150 test examples per class.

HTB's displayed setup standardizes before splitting. This study notebook keeps the same learning objective but splits first, because test examples should not influence training-time preprocessing.


In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

print(f"Training rows: {len(y_train)}; class counts: {np.bincount(y_train)}")
print(f"Test rows:     {len(y_test)}; class counts: {np.bincount(y_test)}")
assert len(y_train) == 1_050
assert len(y_test) == 450
assert np.array_equal(np.bincount(y_train), [350, 350, 350])
assert np.array_equal(np.bincount(y_test), [150, 150, 150])


## 4. Standardize using training data only

For feature `k`, the standardization formula is

`z_ik = (x_ik − μ_k) / σ_k`.

Read it as **“z sub i k equals x sub i k minus mu sub k, divided by sigma sub k.”** `x_ik` is feature `k` of example `i`; `μ_k` is the training-set mean for that feature; `σ_k` is its training-set standard deviation. The result is measured in standard deviations from the mean. For example, if a measurement is 10, its training mean is 8, and its standard deviation is 2, the scaled value is `(10 − 8) / 2 = 1`.

The scaler first **fits** by calculating `μ_k` and `σ_k` from `X_train_raw`, then **transforms** both splits with those same values. `fit_transform` combines those two actions for training data; `transform` reuses the learned values for test data. If the test set helped choose the means and standard deviations, information from evaluation data would leak into training preparation.


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("Training feature means (approximately zero):", X_train.mean(axis=0))
print("Training feature standard deviations (approximately one):", X_train.std(axis=0))
print("Test features transformed with training-only scaler:", X_test.shape)
assert X_train.shape == (1_050, 2)
assert X_test.shape == (450, 2)
assert np.allclose(X_train.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(X_train.std(axis=0), 1.0, atol=1e-12)


## 5. Visualize the clean training examples

Each plotted dot is a training example in the standardized two-feature space. Color indicates its unchanged class. The legend translates the numeric labels back into the manufacturing story. This is a baseline picture before clean-label poisoning; there are no altered features in this section.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for class_id, class_name in CLASS_NAMES.items():
    rows = y_train == class_id
    ax.scatter(
        X_train[rows, 0],
        X_train[rows, 1],
        color=CLASS_COLORS[class_id],
        edgecolors="#141d2b",
        alpha=0.75,
        s=38,
        label=f"Class {class_id}: {class_name}",
    )

ax.set_title("Clean training data — before any feature poisoning")
ax.set_xlabel("Standardized measurement 1")
ax.set_ylabel("Standardized measurement 2")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()


## COAE quick recall — clean-label attack setup

- **Label flipping:** change `y_i`, leave feature row `x_i` fixed.
- **Clean-label poisoning:** subtly change feature row `x_i`, leave label `y_i` unchanged.
- **Goal:** often make a particular inference-time target receive a chosen wrong class.
- **“Clean” means:** the training labels remain plausible and unmodified; feature values may be poisoned.
- **Standardization:** subtract a feature's training mean and divide by its training standard deviation.
- **Leakage-safe preprocessing:** split first, fit the scaler on training features only, transform test features with that fitted scaler.
- **Stratification:** keeps approximate class proportions in train and test partitions.
- **This subsection's scope:** build and inspect the clean three-class dataset. Do not claim the attack has been implemented or succeeded yet.

**Exam check:** If a malicious training example is moved in feature space but keeps its original class label, that is clean-label poisoning. If the class label itself is changed, it is label flipping. Always state whether the attacker altered features, labels, or both.


## 6. Train a clean One-vs-Rest baseline

Binary logistic regression separates one positive group from one negative group. One-vs-Rest (OvR) extends this to `K` classes by fitting `K` binary models. With our `K = 3` labels, the models answer: “Class 0 vs all others,” “Class 1 vs all others,” and “Class 2 vs all others.”

For model `k`, HTB writes the score as `z_k = w_kᵀ x + b_k`. Say **“z sub k equals w sub k transpose times x, plus b sub k.”** `w_k` is that model's learned weight vector, `x` is one feature row, `b_k` is its bias, and `z_k` is its score. The predicted class is `ŷ = arg max_k z_k`: say **“y hat equals the argument k that maximizes z sub k,”** meaning choose the class index whose score is largest. `arg max` asks *which class wins*, rather than returning the winning score itself.

The wrapper stores three separately fitted binary estimators. We inspect their weights and biases so the written equation can be connected to the fitted code. This baseline uses only the original clean training labels. Its accuracy is measured on the untouched test set.


In [ ]:
# Fit one binary logistic-regression model per class: each class vs. the rest.
base_estimator = LogisticRegression(
    random_state=SEED,
    C=1.0,
    solver="liblinear",
)
baseline_model = OneVsRestClassifier(base_estimator)
baseline_model.fit(X_train, y_train)

# The clean test set remains separate from model fitting.
y_pred_baseline = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
print(f"Clean OvR baseline accuracy on the held-out test set: {baseline_accuracy:.4f}")
print(f"Classes in prediction order: {baseline_model.classes_}")

# Each binary estimator stores one weight per feature and one intercept.
class_weights = np.vstack([
    estimator.coef_[0] for estimator in baseline_model.estimators_
])
class_biases = np.array([
    estimator.intercept_[0] for estimator in baseline_model.estimators_
])
for class_id, weights, bias in zip(baseline_model.classes_, class_weights, class_biases):
    print(f"Class {class_id} vs rest: weights={weights}, bias={bias:.4f}")

# Work one example through the linear-score equation z_k = w_k^T x + b_k.
example = X_test[0]
scores_from_formula = class_weights @ example + class_biases
scores_from_model = baseline_model.decision_function(X_test[:1])[0]
winning_class = baseline_model.classes_[np.argmax(scores_from_formula)]
print("\nOne test row (scaled features):", example)
print("Scores computed from weights and bias:", scores_from_formula)
print("Scores returned by the OvR model:     ", scores_from_model)
print("argmax score selects class:           ", winning_class)
assert np.allclose(scores_from_formula, scores_from_model)
assert winning_class == y_pred_baseline[0]


### Connect the numbers to the decision boundary

`estimator.coef_[0]` holds `w_k`, and `estimator.intercept_[0]` holds `b_k`. The code's matrix product `class_weights @ example + class_biases` calculates one `z_k` score for each class. `np.argmax` returns the position of the largest score; `baseline_model.classes_` maps that position back to the actual class label. The assertions confirm that this hand-written score calculation agrees with the OvR model for the example.

Two classes `i` and `j` have equal scores at their shared boundary: `z_i = z_j`. Substituting the score equation and moving everything to the left gives `(w_i − w_j)ᵀ x + (b_i − b_j) = 0`. Say **“w sub i minus w sub j, transpose, times x, plus b sub i minus b sub j, equals zero.”** In two dimensions it is a line. For example, if `z_0 = 2x_1 + x_2 − 1` and `z_1 = x_1 + 2x_2`, their tie line is `x_1 − x_2 − 1 = 0`; at `(2, 1)`, both scores are 4.


In [ ]:
# Evaluate the fitted classifier on a grid to see which class wins where.
# A 0.04 step keeps this teaching plot responsive while showing clear regions.
mesh_step = 0.04
x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.arange(x_min, x_max, mesh_step),
    np.arange(y_min, y_max, mesh_step),
)
mesh_points = np.column_stack((xx.ravel(), yy.ravel()))
region_predictions = baseline_model.predict(mesh_points).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 7))
region_cmap = plt.cm.colors.ListedColormap([
    CLASS_COLORS[class_id] for class_id in baseline_model.classes_
])
ax.contourf(xx, yy, region_predictions, levels=[-0.5, 0.5, 1.5, 2.5], cmap=region_cmap, alpha=0.22)
for class_id, class_name in CLASS_NAMES.items():
    rows = y_train == class_id
    ax.scatter(
        X_train[rows, 0], X_train[rows, 1],
        color=CLASS_COLORS[class_id], edgecolors="#141d2b",
        alpha=0.78, s=30, label=f"Class {class_id}: {class_name}",
    )
ax.set_title(f"Clean One-vs-Rest baseline (test accuracy {baseline_accuracy:.3f})")
ax.set_xlabel("Standardized measurement 1")
ax.set_ylabel("Standardized measurement 2")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()


## COAE quick recall — OvR baseline

- **Purpose:** establish the clean model's behavior before poisoning, so later changes have a reference point.
- **OvR:** for `K` classes, train `K` binary models; model `k` separates class `k` from all remaining classes.
- **Score:** `z_k = w_kᵀ x + b_k`, read **“z sub k equals w sub k transpose times x, plus b sub k.”** The weights multiply the input features; the bias shifts the score.
- **Prediction:** `ŷ = arg max_k z_k`, read **“y hat is the class index that maximizes z sub k.”** Choose the class with the highest score.
- **Boundary:** two class models tie where `z_i = z_j`; rearranging gives `(w_i − w_j)ᵀ x + (b_i − b_j) = 0`.
- **Model-to-code map:** `coef_` stores weights, `intercept_` stores bias, `np.argmax` selects the winning score.
- **Evaluation:** fit on clean training data and compare later attacks on the same clean held-out test set.
- **Reference result:** HTB reports 0.9600 accuracy for its seeded synthetic split; the baseline is a dataset-specific benchmark.

**Exam check:** For three classes, OvR uses three binary models. If their scores are `(0.2, 1.7, −0.4)`, Class 1 wins. A point is on the boundary between classes `i` and `j` when their scores are equal. The baseline itself is not an attack and does not prove robustness.


## 7. Identify a correctly classified Class 1 target near the Class 0 boundary

The next clean-label attack stage needs a specific training example to target. We select a real Class 1 row that the clean baseline still predicts as Class 1, then find the eligible row nearest the Class 0/Class 1 score tie. No features or labels are changed here.

For a feature vector `x`, the two baseline scores are `z_0(x)` and `z_1(x)`. HTB defines their difference as `f_01(x) = z_0(x) - z_1(x) = (w_0 - w_1)^T x + (b_0 - b_1)`. Read this aloud as **“f zero-one of x equals z zero of x minus z one of x, which equals w zero minus w one, transpose, times x, plus b zero minus b one.”** The subscript `01` names the pair of classes; the superscript `T` means transpose and, here, gives the dot product between a weight vector and the feature vector.

If `f_01(x) < 0`, then `z_1(x) > z_0(x)`: Class 1 wins that pairwise comparison. If the value is near zero, the two scores are close. We only consider rows whose true label is 1 and whose full baseline prediction is also 1, so the target is both genuinely Class 1 and correctly classified before any attack.

Among those eligible rows, choose the one with the largest (least negative) `f_01` value. For example, `−0.02` is larger than `−0.30` and is nearer to the tie value zero. `np.argmax` returns the position of the largest value; it does not return the score itself. We map that position back to the original training-row index.

**Geometry connection:** for this fixed linear boundary, perpendicular distance is `|f_01(x)| / ||w_0 - w_1||_2`. Say **“absolute f zero-one of x divided by the L-two norm of w zero minus w one.”** The denominator is the same for every point on this one boundary, so among correctly classified Class 1 points, minimizing the absolute score difference also minimizes distance. For example, a score difference of `−0.2` with a weight-difference norm of `2` is `0.1` distance units from the line.

The equation is a pairwise comparison. A third class exists, so the full `baseline_model.predict` check is retained as a safety check: being on the Class 1 side of the 0/1 tie alone does not guarantee that Class 1 beats Class 2 too.


In [ ]:
# Formula implemented: f_01(x) = (w_0 - w_1)^T x + (b_0 - b_1).
# First locate the 0 and 1 rows in the fitted model's class ordering instead
# of assuming an estimator array position without checking the class labels.
class0_position = int(np.flatnonzero(baseline_model.classes_ == 0)[0])
class1_position = int(np.flatnonzero(baseline_model.classes_ == 1)[0])
w_diff_01 = class_weights[class0_position] - class_weights[class1_position]
b_diff_01 = class_biases[class0_position] - class_biases[class1_position]

# Keep only true Class 1 training rows. `candidate_indices` stores their row
# positions in the complete, clean training split; labels remain untouched.
candidate_indices = np.flatnonzero(y_train == 1)
X_class1_train = X_train[candidate_indices]

# The matrix-vector product computes (w_0 - w_1)^T x for every candidate.
# Adding the scalar bias difference completes the pairwise score formula.
f_01_values = X_class1_train @ w_diff_01 + b_diff_01
candidate_predictions = baseline_model.predict(X_class1_train)

# Eligibility requires BOTH: negative pairwise margin (Class 1 beats Class 0)
# and a full OvR prediction of Class 1 (Class 1 also beats Class 2).
eligible_positions = np.flatnonzero(
    (f_01_values < 0) & (candidate_predictions == 1)
)
if len(eligible_positions) == 0:
    raise RuntimeError(
        "No correctly classified Class 1 training examples lie on the Class 1 "
        "side of the Class 0/Class 1 boundary. Inspect the baseline before continuing."
    )

# `argmax` returns a position within eligible_positions. Because every eligible
# score is negative, the largest score is the one closest to zero.
closest_eligible_position = eligible_positions[
    np.argmax(f_01_values[eligible_positions])
]
target_train_index = int(candidate_indices[closest_eligible_position])
X_target = X_train[target_train_index].copy()
y_target = int(y_train[target_train_index])
target_baseline_prediction = int(
    baseline_model.predict(X_target.reshape(1, -1))[0]
)
target_f_01 = float(f_01_values[closest_eligible_position])
weight_difference_norm = float(np.linalg.norm(w_diff_01, ord=2))
if weight_difference_norm == 0:
    raise RuntimeError("The Class 0/Class 1 score functions have no separating direction.")

# Geometry formula: distance to the 0/1 tie line = |f_01(x)| / ||w_0-w_1||_2.
target_boundary_distance = abs(target_f_01) / weight_difference_norm

print("Selected clean training row:", target_train_index)
print("Target features (standardized):", X_target)
print("True label / baseline prediction:", y_target, "/", target_baseline_prediction)
print(f"Class 0 minus Class 1 score, f_01(x): {target_f_01:.6f}")
print(f"Perpendicular distance to 0/1 boundary: {target_boundary_distance:.6f}")
print(f"Eligible correctly classified Class 1 rows: {len(eligible_positions)}")

# Confirm the selection properties before visualizing or using it later.
assert y_target == 1
assert target_baseline_prediction == 1
assert target_f_01 < 0
assert np.isfinite(target_boundary_distance)
assert np.all(candidate_predictions[eligible_positions] == 1)
assert target_f_01 == np.max(f_01_values[eligible_positions])


### See the selected point in the clean training data

The white cross marks the selected row. It is still an unmodified Class 1 point. Its location near the Class 0 region makes it a useful teaching target because only a small change in the learned boundary may change which of those two classes wins. The plot is a visual check of the target-selection step, not evidence that poisoning has happened.


In [ ]:
# Reuse the clean class colors and plot the selected point above the other dots.
fig, ax = plt.subplots(figsize=(10, 7))
for class_id, class_name in CLASS_NAMES.items():
    rows = y_train == class_id
    ax.scatter(
        X_train[rows, 0],
        X_train[rows, 1],
        color=CLASS_COLORS[class_id],
        edgecolors="#141d2b",
        alpha=0.72,
        s=32,
        label=f"Class {class_id}: {class_name}",
    )

ax.scatter(
    X_target[0],
    X_target[1],
    marker="P",
    s=230,
    facecolor="white",
    edgecolor="#141d2b",
    linewidth=1.8,
    label=f"Selected target: train row {target_train_index}",
    zorder=5,
)
ax.set_title("Clean training data with the Class 1 target highlighted")
ax.set_xlabel("Standardized measurement 1")
ax.set_ylabel("Standardized measurement 2")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()


## COAE quick recall — identifying a target

- **Target candidate:** a genuine Class 1 training example; this section does not edit it.
- **Pairwise score difference:** `f_01(x) = z_0(x) - z_1(x) = (w_0-w_1)^T x+(b_0-b_1)`. Read it as **“Class 0's score minus Class 1's score.”**
- **Side of boundary:** `f_01(x) < 0` means `z_1(x) > z_0(x)`, so Class 1 beats Class 0.
- **Near boundary:** a negative value close to zero means the two scores are close. Among negative values, `−0.02` is larger and closer to zero than `−0.30`.
- **Selection:** filter to true Class 1 rows correctly predicted as Class 1, then use `argmax` on their negative `f_01` values. `argmax` returns the row position with the largest score.
- **Distance connection:** perpendicular distance to the linear tie boundary is `|f_01(x)| / ||w_0-w_1||_2`; for a fixed boundary, smallest absolute score difference means nearest point.
- **Three-class check:** the 0/1 margin only compares Classes 0 and 1; verify the complete classifier still predicts Class 1.
- **Scope:** selection only. The training feature row and label have not been changed and no poisoned model has been trained.

**Exam check:** If a correctly classified Class 1 point has `f_01=−0.04` and another has `f_01=−0.40`, select `−0.04` as the point closer to the Class 0/Class 1 boundary. State that its true label and baseline prediction are both still Class 1.
